# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) protein abundance dataset using the `mlcroissant` library, referencing all entities by their Croissant schema `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library and visualization tools are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

## View key metadata properties
print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", dataset.metadata.version)
print("License:", dataset.metadata.license)
print("Citation:", dataset.metadata.citeAs)

## 2. Data Overview
Review all available record sets, their fields, and columns. All entities are referenced by their `@id`.

In [ ]:
# List all record sets with their @id and labels
record_sets = dataset.record_sets
print(f"Record sets found: {len(record_sets)}\n-----------------------")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'Unnamed')}")
    if 'field' in rs and isinstance(rs['field'], list):
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")
    print()

In [ ]:
# Show a record from each record set
for rs in record_sets:
    record_set_id = rs['@id']
    print(f"Records from record set @id: {record_set_id} (showing one example):")
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            print(record)
            if i >= 0:  # show only one record for brevity
                break
    except Exception as e:
        print(f"  Could not load records: {e}")
    print("-")


## 3. Data Extraction
Load all data from all record sets into pandas DataFrames keyed by their record set `@id`s for analysis.

In [ ]:
# Get all record set @id's for batch extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Try to extract records
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for {record_set_id} | shape: {dataframes[record_set_id].shape}")
            print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
            print(dataframes[record_set_id].head(2))
        else:
            print(f"No records found for record set @id {record_set_id}")
    except Exception as e:
        print(f"Could not extract DataFrame for {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply standard data cleaning and transformation operations on a key numeric field. All field and column references by their Croissant schema `@id`.


In [ ]:
# Select one record set with tabular data for EDA
# For this FAIR^2 dataset, we look for one with protein quantifications. For example, suppose the main record set contains a field '@id' like 'cr:proteinTable' and numeric columns like '@id': 'cr:molecularWeight' etc.

# For illustration, let's use the first available tabular record set
main_tabular_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        # Find a DataFrame with at least one float/integer column
        numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
        if numeric_cols:
            main_tabular_record_set_id = rsid
            print(f"Selected record set for EDA: {main_tabular_record_set_id}")
            print(f"Numeric fields by column name: {numeric_cols}")
            break
        else:
            print(f"No numeric fields found in {rsid}.")

if not main_tabular_record_set_id:
    raise ValueError("No suitable tabular record set with numeric fields found.")

df = dataframes[main_tabular_record_set_id]
# Choose the first numeric column for demonstration
numeric_field_id = numeric_cols[0]
print(f"Analyzing numeric field (column): {numeric_field_id}")

threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head(3))

# Normalize the field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Group by a categorical field, e.g., '@id' containing 'description' or similar
possible_group_fields = [col for col in df.columns if df[col].dtype == object]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"\nGrouping by {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    print("No categorical text columns for grouping found.")

## 5. Visualization
Visualize the distribution of a key numeric field and its relationship to a group or category.


In [ ]:
# Distribution plot of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping is meaningful, plot group means for the numeric field
if possible_group_fields:
    plt.figure(figsize=(10,5))
    # Show only top 10 groups by count
    top_groups = df[group_field].value_counts().head(10).index
    toplot = df[df[group_field].isin(top_groups)]
    sns.boxplot(x=group_field, y=numeric_field_id, data=toplot)
    plt.title(f"{numeric_field_id} by {group_field} (Top 10)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We have loaded, overviewed, and explored the FAIR^2 mass spectrometry dataset via its Croissant schema.
- All references to fields and entities use their canonical Croissant `@id`s, ensuring reproducibility and schema alignment.
- Demonstrated basic EDA on a core dataset with emphasis on numeric protein abundance fields using standardized Croissant conventions.
- This approach ensures interoperable, fully-documented ML data pipelines and richness in FAIR metadata for downstream use.